# Self-Correcting Code Generator | Reflection/Self-Correction

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class ReflectionState(TypedDict):
    input: str
    draft: NotRequired[str]
    critique: NotRequired[str]
    iteration: NotRequired[int]
    final_output: NotRequired[str]

In [5]:
MAX_ITERATIONS = 3

In [6]:
# Generator: produce or refine content
def generate(state: ReflectionState) -> dict:
    if state.get("critique"):
        prompt = (
            f"Revise your previous draft based on this feedback.\n\n"
            f"Original request: {state['input']}\n"
            f"Previous draft:\n{state['draft']}\n"
            f"Feedback:\n{state['critique']}"
        )
    else:
        prompt = f"Write a Python function for: {state['input']}"

    response = model.invoke(prompt)
    iteration = state.get("iteration", 0) + 1
    return {"draft": response.content, "iteration": iteration}

# Critic: evaluate, provide feedback, and route via Command
def reflect(state: ReflectionState) -> Command[Literal["generate", "finalize"]]:
    response = model.invoke(
        f"Review this code for correctness, edge cases, and best practices. "
        f"If it's good, respond with exactly 'APPROVED'. "
        f"Otherwise, provide specific feedback.\n\n{state['draft']}"
    )
    critique = response.content
    iteration = state.get("iteration", 0)
    if critique.strip() == "APPROVED" or iteration >= MAX_ITERATIONS:
        return Command(goto="finalize", update={"critique": critique})
    return Command(goto="generate", update={"critique": critique})

def finalize(state: ReflectionState) -> dict:
    return {"final_output": state["draft"]}

In [7]:
# Build graph
graph = StateGraph(ReflectionState)
graph.add_node("generate", generate)
graph.add_node("reflect", reflect)
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate")
graph.add_edge("generate", "reflect")
# No add_conditional_edges needed -- reflect returns Command to route directly
graph.add_edge("finalize", END)

reflection_agent = graph.compile()

In [8]:
plot_mermaid(reflection_agent)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	reflect(reflect)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	generate --> reflect;
	reflect -.-> finalize;
	reflect -.-> generate;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
result = reflection_agent.invoke({
    "input": "a function that finds all prime numbers up to N using the Sieve of Eratosthenes"
})
print(result["final_output"])

The improvements suggested for the Sieve of Eratosthenes function address crucial aspects such as handling edge cases and enhancing code readability. Here's the refined version of the function implementing those suggestions:

### Revised Function:

```python
def sieve_of_eratosthenes(N):
    # Handle edge case where N is less than 2
    if N < 2:
        return []
    
    # Initialize a boolean array where each entry represents if the number is prime
    is_prime = [True] * (N + 1)
    p = 2
    
    while p * p <= N:
        # If is_prime[p] is True, it is a prime number
        if is_prime[p]:
            # Mark all multiples of p as non-prime
            for i in range(p * p, N + 1, p):
                is_prime[i] = False
        p += 1

    # Collect numbers that are marked as prime
    prime_numbers = [p for p in range(2, N + 1) if is_prime[p]]
    return prime_numbers

# Example usage:
N = 30
print(sieve_of_eratosthenes(N))  # Output: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
```

##

In [10]:
stream_invoke(reflection_agent, {
    "input": "a function that finds all prime numbers up to N using the Sieve of Eratosthenes"
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'input': 'a function that finds all prime numbers up to N using the Sieve of Eratosthenes',
 'draft': "The improvements suggested for the Sieve of Eratosthenes function address crucial aspects such as handling edge cases and enhancing code readability. Here's the refined version of the function implementing those suggestions:\n\n### Revised Function:\n\n```python\ndef sieve_of_eratosthenes(N):\n    # Handle edge case where N is less than 2\n    if N < 2:\n        return []\n    \n    # Initialize a boolean array where each entry represents if the number is prime\n    is_prime = [True] * (N + 1)\n    p = 2\n    \n    while p * p <= N:\n        # If is_prime[p] is True, it is a prime number\n        if is_prime[p]:\n            # Mark all multiples of p as non-prime\n            for i in range(p * p, N + 1, p):\n                is_prime[i] = False\n        p += 1\n\n    # Collect numbers that are marked as prime\n    prime_numbers = [p for p in range(2, N + 1) if is_prime[p]]\n    retur